In [1]:
import re
import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [2]:
# ── STEP 1: Dataset (10 movies) ────────────────────────────────────────
movies = [
    {"title": "Interstellar",
     "description": "A team of astronauts travels through a wormhole in space to find a new home for humanity."},
    {"title": "Star Wars",
     "description": "A space adventure where rebels and robots battle an evil galactic empire."},
    {"title": "Wall-E",
     "description": "A small robot is left alone on Earth and falls in love while exploring space."},
    {"title": "The Martian",
     "description": "An astronaut is stranded alone on Mars and uses science to survive in space."},
    {"title": "Avatar",
     "description": "A soldier explores an alien jungle planet and joins the native creatures to protect nature."},
    {"title": "Ratatouille",
     "description": "A rat with a passion for cooking secretly helps a chef create amazing food in a Paris restaurant."},
    {"title": "Julie and Julia",
     "description": "A woman learns to cook hundreds of recipes and writes a food blog inspired by a famous chef."},
    {"title": "The Dark Knight",
     "description": "A superhero battles a criminal mastermind who brings chaos and crime to Gotham City."},
    {"title": "Inception",
     "description": "A thief enters people's dreams to steal secrets from their minds in a surreal heist."},
    {"title": "Chef",
     "description": "A talented chef loses his job and starts a food truck to rediscover his love for cooking."},
]

In [3]:
# ── STEP 2: Preprocessing ──────────────────────────────────────────────
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text   = text.lower()                               # lowercase
    text   = re.sub(r'[^\w\s]', '', text)              # remove punctuation
    tokens = text.split()                               # split into words
    tokens = [w for w in tokens if w not in stop_words] # remove stopwords
    tokens = [lemmatizer.lemmatize(w) for w in tokens] # lemmatize
    return ' '.join(tokens)

documents_clean = [preprocess(m['description']) for m in movies]
titles          = [m['title'] for m in movies]
descriptions    = [m['description'] for m in movies]

In [4]:
# ── STEP 3: TF-IDF Matrix ──────────────────────────────────────────────
vectorizer   = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents_clean)

# ── STEP 4 + 5 + 6: Search Function ───────────────────────────────────
def search(query, top_n=3):
    print(f"\n{'='*55}")
    print(f"  Query: '{query}'")
    print(f"{'='*55}")

    query_clean  = preprocess(query)          # same preprocessing as docs
    query_vector = vectorizer.transform([query_clean])  # transform (NOT fit!)
    scores       = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices  = np.argsort(scores)[::-1][:top_n]   # sort desc, take top N

    print(f"\n  Top {top_n} Results:")
    print(f"  {'-'*52}")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  #{rank}  {titles[idx]}")
        print(f"      Score      : {round(scores[idx], 4)}")
        print(f"      Description: {descriptions[idx]}")
        print()


In [5]:
search("space adventure with robots")
search("healthy food and cooking")
search("crime and action hero")


  Query: 'space adventure with robots'

  Top 3 Results:
  ----------------------------------------------------
  #1  Star Wars
      Score      : 0.5602
      Description: A space adventure where rebels and robots battle an evil galactic empire.

  #2  Wall-E
      Score      : 0.2862
      Description: A small robot is left alone on Earth and falls in love while exploring space.

  #3  The Martian
      Score      : 0.1134
      Description: An astronaut is stranded alone on Mars and uses science to survive in space.


  Query: 'healthy food and cooking'

  Top 3 Results:
  ----------------------------------------------------
  #1  Chef
      Score      : 0.3862
      Description: A talented chef loses his job and starts a food truck to rediscover his love for cooking.

  #2  Ratatouille
      Score      : 0.3603
      Description: A rat with a passion for cooking secretly helps a chef create amazing food in a Paris restaurant.

  #3  Julie and Julia
      Score      : 0.154
      D